# RG3 11개에서 점진 축소
각 고정 조합은 내부 정상 기준 재보정. 개발 비교.

In [1]:
from pathlib import Path
import json,itertools,nbformat
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import NearestNeighbors
ROOT=Path.cwd();OUT=ROOT/'reports/tables'
src='\n'.join(c.source for c in nbformat.read('notebooks/187_rg3_gas_max3_classifier_refit.ipynb',4).cells if c.cell_type=='code')
exec(src[:src.index('memo={};audits=[]')])
source54=''.join(c.source for c in nbformat.read('notebooks/54_rg3_pdf_thermal_confirm_gates.ipynb',4).cells if c.cell_type=='code')
F={'pressure':pd.read_csv(OUT/'rg3_missed11_pressure_transition_features.csv').drop(columns='TimeStamp')}
controls={'pressure':['Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3']}
exec(source54[source54.index('def local_residual('):source54.index('memo={}')])
local=local_residual
headnames=['RF42','RF7','RF101','ET42','ET7','ET101','LR01','LR1','LR10']
sets={
 'original11':headnames,
 '10_drop_LR10':headnames[:-1],
 '9_LR1_only':headnames[:6]+['LR1'],
 '8_trees_only':headnames[:6],
 '9_two_seeds_LR3':['RF42','RF7','ET42','ET7','LR01','LR1','LR10'],
 '7_two_seeds_LR1':['RF42','RF7','ET42','ET7','LR1'],
 '6_two_seeds_trees':['RF42','RF7','ET42','ET7'],
 '5_one_seed_each':['RF42','ET42','LR1'],
}
cache={};audit=[]
def heads(ref,q):
 key=(tuple(ref),tuple(q))
 if key in cache:return cache[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 audit.append(dict(train=len(ref),query=len(q),overlap=len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q]))))
 if len(np.unique(y[ref]))<2:return np.zeros((len(q),9))
 tr=local(ref,ref,'pressure');te=local(ref,q,'pressure')
 imp=SimpleImputer(keep_empty_features=True).fit(tr);tr=imp.transform(tr);te=imp.transform(te)
 out=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  for seed in [42,7,101]:
   m=cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2).fit(tr,y[ref])
   out.append(m.predict_proba(te)[:,1])
 sc=RobustScaler().fit(tr[y[ref]==0])
 for C in [.1,1.,10.]:
  m=LogisticRegression(C=C,class_weight='balanced',max_iter=2000,random_state=42).fit(sc.transform(tr),y[ref])
  out.append(m.predict_proba(sc.transform(te))[:,1])
 cache[key]=np.column_stack(out);return cache[key]
def score(a,names):
 families=[]
 for prefix in ['RF','ET','LR']:
  ids=[headnames.index(n) for n in names if n.startswith(prefix)]
  if ids:families.append(a[:,ids].mean(axis=1))
 return np.mean(families,axis=0)
def direction(ids):
 a=d.iloc[ids]
 return ((a.Max_Injection_Pressure__delta1.round(2)<0)&(a.Max_Injection_Speed__delta1.round(2)>0)&(a.Max_Switch_Over_Pressure__delta1.round(2)<a.Max_Injection_Pressure__delta1.round(2))).to_numpy()
base=pd.read_csv(OUT/'rg3_187_small_classifier_predictions.csv')
base=base[base.policy.eq('reference_RF2_original')].set_index('TimeStamp')
anchor=pd.read_csv(OUT/'rg3_pressure_seed_majority_predictions.csv')
anchor=anchor[(anchor.scheme=='leave_one_block_out') & (anchor.anchor=='exploratory54')].set_index('TimeStamp')
rows=[]
for held in blocks:
 q=np.flatnonzero(groups.eq(held));ref=np.flatnonzero(~groups.eq(held))
 outer=heads(ref,q);inner=np.full((len(ref),9),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy()
  inner[mask]=heads(ref[~mask],ref[mask])
 for label,names in sets.items():
  si=score(inner,names);so=score(outer,names)
  cut=float(np.quantile(si[y[ref]==0],.99,method='higher'))
  b=base.loc[d.TimeStamp.iloc[q].astype(str),'prediction'].to_numpy(bool)
  pr=b|((so>cut)&direction(q))
  for j,idx in enumerate(q):rows.append(dict(candidate=label,classifiers=2+len(names),TimeStamp=d.TimeStamp.iloc[idx],block=held,actual=int(y[idx]),prediction=int(pr[j]),pressure_score=float(so[j]),cutoff=cut))
  fullanchor=anchor.loc[d.TimeStamp.iloc[q].astype(str),'anchor_prediction'].to_numpy(bool)
  joined=fullanchor|((so>cut)&direction(q))
  for j,idx in enumerate(q):rows.append(dict(candidate='preserved_anchor23_'+label,classifiers=23+len(names),TimeStamp=d.TimeStamp.iloc[idx],block=held,actual=int(y[idx]),prediction=int(joined[j]),pressure_score=float(so[j]),cutoff=cut))
 pd.DataFrame(rows).to_csv(OUT/'rg3_191_pressure_head_reduction_progress.csv',index=False)
 print('Finished',held,flush=True)
pp=pd.DataFrame(rows);metrics=[]
for label,g in pp.groupby('candidate',sort=False):
 assert len(g)==583 and g.TimeStamp.is_unique
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 metrics.append(dict(candidate=label,classifiers=int(g.classifiers.iloc[0]),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=float(2*tp/(2*tp+fp+fn))))
mm=pd.DataFrame(metrics).sort_values('classifiers',ascending=False)
old=pd.read_csv(OUT/'rg3_190_gradual_branch_ablation_predictions.csv')
old=old[old.candidate.eq('I0_T0_seed42_7_101_vote1')]
check=pp[pp.candidate.eq('original11')].merge(old[['TimeStamp','prediction']].assign(TimeStamp=lambda a:pd.to_datetime(a.TimeStamp)),on='TimeStamp',suffixes=('_new','_old'))
assert len(check)==583
mismatch=int(check.prediction_new.ne(check.prediction_old).sum())
assert mismatch==0
oldpressure=pd.read_csv(OUT/'rg3_pressure_direction_predictions.csv')
oldpressure=oldpressure[(oldpressure.scheme=='leave_one_block_out') & (oldpressure.anchor=='exploratory54') & (oldpressure.model=='pressure_fall_speed_rise_switch_steeper')]
verify=pp[pp.candidate.eq('preserved_anchor23_original11')].merge(oldpressure[['TimeStamp','prediction','score','threshold']].assign(TimeStamp=lambda a:pd.to_datetime(a.TimeStamp)),on='TimeStamp',suffixes=('_new','_old'))
assert len(verify)==583 and verify.prediction_new.eq(verify.prediction_old).all()
assert np.allclose(verify.pressure_score,verify.score,atol=1e-12) and np.allclose(verify.cutoff,verify.threshold,atol=1e-12)
check[check.prediction_new.ne(check.prediction_old)].to_csv(OUT/'rg3_191_reference_mismatches.csv',index=False)
print('Original11 prediction mismatches:',mismatch)
pp.to_csv(OUT/'rg3_191_pressure_head_reduction_predictions.csv',index=False,encoding='utf-8-sig')
mm.to_csv(OUT/'rg3_191_pressure_head_reduction_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audit).to_csv(OUT/'rg3_191_pressure_head_reduction_fit_audit.csv',index=False)
assert all(a['overlap']==0 for a in audit)
print(mm.to_string(index=False))
text='# RG3 11개 내부 모델 점진 축소\n\n정상566/가스17, 동일5구간 제외 검증. 각 축소 조합은 학습 내부 정상 점수99%로 임계값을 다시 계산했다. 저장된 원본11개와 예측 불일치 '+str(mismatch)+'개: 완전 재현으로 주장하지 않는다.\n\n'+mm.to_string(index=False)
text+='\n\n## 모델 구성\n'+json.dumps(sets,ensure_ascii=False,indent=2)
text+='\n\n수정 원인: 190번이 압력 prediction에 포함된 anchor를 압력 단독으로 오인했다. 수정 후 RF2+압력9의11개는TP5/FP6이며 완전 재현된다. preserved_anchor23은기존RF2+패턴RF12+열9의저장된구간제외출력을보존한다. 압력9개는실제로재학습해기존score/cut 및anchor결합583예측모두일치했다. 다른23개는이번에재학습하지않았다.\nLR 숫자는C값, RF/ET 숫자는random seed다. RF2 정상 관계 피처에는Ridge4 전처리가 별도 필요하다.\n\n## 한계\n미리 명시한 축소 조합들의 개발 비교다. 평가 결과를 보고 최종 조합을 선택하면 독립검증 성능으로 부를 수 없다. 기존 이력 피처 조건을 재사용했으며 미관측 미래Run 검증이 아니다. 비라벨 경보율 확인 및 새 저장 추론 모델 교체는 하지 않았다.\n'
(ROOT/'reports/rg3_191_pressure_head_reduction.md').write_text(text,encoding='utf-8')



C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\nbformat\validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


Finished 20201021_001626_RG3


Finished 20201022_002405_RG3


Finished 20201022_044026_RG3


Finished 20201023_010128_RG3


Finished 20201023_044148_RG3


Original11 prediction mismatches: 0
                           candidate  classifiers  TP  FP  FN  TN       F1
       preserved_anchor23_original11           32   8   6   9 560 0.516129
     preserved_anchor23_10_drop_LR10           31   8   6   9 560 0.516129
       preserved_anchor23_9_LR1_only           30   8   6   9 560 0.516129
  preserved_anchor23_9_two_seeds_LR3           30   7   6  10 560 0.466667
     preserved_anchor23_8_trees_only           29   7   6  10 560 0.466667
  preserved_anchor23_7_two_seeds_LR1           28   7   6  10 560 0.466667
preserved_anchor23_6_two_seeds_trees           27   7   6  10 560 0.466667
  preserved_anchor23_5_one_seed_each           26   8   6   9 560 0.516129
                          original11           11   5   6  12 560 0.357143
                        10_drop_LR10           10   5   6  12 560 0.357143
                     9_two_seeds_LR3            9   4   6  13 560 0.296296
                          9_LR1_only            9   5   6  12 56

2615